In [ ]:
import pandas as pd

In [ ]:
# load FORUM scores for all policies & discussions
q_df = []
for ncomments in ['10', 'N']:
    for feature in ['lexdiv', 'sentcomp', 'sim', 'smog']:
        if ncomments == '10':
            c = 1
        elif ncomments == 'N':
            c = 4
        else:
            raise ValueError("Invalid value for ncomments")

        # Read the data from CSV file
        df = pd.read_csv(f'model_output/forum-scores/df_{feature}_list_{c}.csv', index_col=0)
        df = df.reset_index().melt(id_vars='index')
        df['n'] = ncomments
        df['feature'] = feature
        q_df.append(df)

q_df = pd.concat(q_df)
q_df.head()

q_df['policy'] = q_df['variable'].str.split('_').str[0]
q_df['replies'] = q_df['variable'].str.split('_').str[-1]
q_df['pinned'] = q_df['variable'].str.contains('pinned').astype(int)

# Create dummy variables for policy, replies, and discussion
policy_dummies = pd.get_dummies(q_df['policy'], prefix='policy')
replies_dummies = pd.get_dummies(q_df['replies'], prefix='replies')

# Concatenate the dummy variables with the original DataFrame
q_df = pd.concat([q_df, policy_dummies, replies_dummies], axis=1)

# Drop unnecessary columns
q_df = q_df.rename(columns={'index': 'discussion', 'variable': 'sorting policy'})

# convert bool cols to int
boolcols = q_df.columns[q_df.dtypes == bool]
q_df[boolcols] = q_df[boolcols].astype(int)

q_df

In [ ]:
# Save the data
q_df.to_csv('data/q_df.csv', index=False)